In [ ]:
import numpy as np
import matplotlib.pyplot as plt 

import pandas as pd  
import seaborn as sns 
from sklearn import metrics
import scipy.io as sio
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
import warnings
warnings.filterwarnings('ignore')
import xarray as xr
from sklearn.preprocessing import PolynomialFeatures

In [ ]:
## training data

week = '2'

file_path_obs_train = f'IMD_Data/Week{week}/IMD_pra_1989-2010.nc'
file_path_model_train = f'IMD_Data/Week{week}/GEFS_pra_1989-2010.nc'
data_obs_train = xr.open_dataset(file_path_obs_train)
data_model_train = xr.open_dataset(file_path_model_train)

## validation data

file_path_obs_test = f'IMD_Data/Week{week}/IMD_pra_2011-2018.nc'
file_path_model_test = f'IMD_Data/Week{week}/GEFS_pra_2011-2018.nc'
data_obs_test = xr.open_dataset(file_path_obs_test)
data_model_test = xr.open_dataset(file_path_model_test)

In [ ]:
## test data

file_path_obs_test_2022 = f'IMD_Data_Test/Week {week}/IMD_pra_2022-2022.nc'
file_path_model_test_2022 = f'IMD_Data_Test/Week {week}/GEFS_pra_2022-2022.nc'
data_obs_test_2022 = xr.open_dataset(file_path_obs_test_2022)
data_model_test_2022 = xr.open_dataset(file_path_model_test_2022)

In [ ]:
test_obs_values_2022 = data_obs_test_2022['prcp'].values[59:181,:,:]
test_model_values_2022 = data_model_test_2022.mean('M')['prcp'].values[59:181,:,:]

In [ ]:
test_obs_values_2022 = test_obs_values_2022[~np.isnan(test_model_values_2022[:,2,10]),:,:]
test_model_values_2022 = test_model_values_2022[~np.isnan(test_model_values_2022[:,2,10]),:,:]

In [ ]:
test_model_values_2022 *= 86400

In [ ]:
train_obs_values = data_obs_train['prcp'].values
train_obs_values.shape

In [ ]:
test_obs_values = data_obs_test['prcp'].values
test_obs_values.shape

In [ ]:
y = np.concatenate([train_obs_values, test_obs_values, test_obs_values_2022], axis=0)
y.shape

In [ ]:
train_model_values = data_model_train.mean('M')['prcp'].values
train_model_values.shape

In [ ]:
test_model_values = data_model_test.mean('M')['prcp'].values
test_model_values.shape

In [ ]:
X = np.concatenate([train_model_values, test_model_values, test_model_values_2022], axis=0)
X.shape

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_t, X_test , y_t, y_test= train_test_split(X, y, test_size=0.1, shuffle=True, random_state=42)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_t, y_t, test_size=0.2, shuffle=True, random_state=42)

In [ ]:
def rps_clim(class_value):
    
    if len(class_value.shape)>1:
        cum_obs = np.cumsum(class_value, axis=1)
        cum_clim_probs = np.cumsum(np.ones((class_value.shape[0],3))*0.3333, axis=1).round(3)
        values = ((cum_clim_probs-cum_obs)**2)/(class_value.shape[1]-1)
    else:
        cum_obs = np.cumsum(class_value, axis=0)
        cum_clim_probs = np.cumsum(np.ones(3)*0.3333, axis=0).round(3)
        values = ((cum_clim_probs-cum_obs)**2)/(class_value.shape[0]-1)
    
    
    if len(class_value.shape)>1:
        return np.mean(np.sum(values, axis=1))
    else:
        return np.sum(values)
    
def rps_forecast(class_value, probs, grid_point=None):
    """
    grid_point: optional if only a single point is being fed as an input
    """
    if grid_point is not None:
        cum_obs = np.cumsum(class_value, axis=1)
        cum_fc_probs = np.cumsum(probs[grid_point[0], grid_point[1]], axis=1).round(3)
        values = ((cum_fc_probs-cum_obs)**2)/(class_value.shape[1]-1)
    else:
        cum_obs = np.cumsum(class_value, axis=1)
        cum_fc_probs = np.cumsum(probs, axis=1).round(3)
        values = ((cum_fc_probs-cum_obs)**2)/(class_value.shape[1]-1)
        
    
    if len(class_value.shape)>1:
        return np.mean(np.sum(values, axis=1))
    else:
        return np.sum(values)
    
def make_ohe(array):
    p_33 = np.quantile(array, q=1/3)
    p_67 = np.quantile(array, q=2/3)
    
    ohe = np.zeros((array.shape[0],3))
    for i in range(len(array)):
        if array[i]<=p_33:
            ohe[i][0] = 1
        elif p_33<array[i]<=p_67:
            ohe[i][1] = 1
        elif array[i]>p_67:
            ohe[i][2] = 1
    
    return ohe, p_33, p_67

In [ ]:
from skgarden import RandomForestQuantileRegressor

In [ ]:
def get_tercile_predictions_single_grid(observed_data_train, model_output_train, observed_data_test, model_output_test,
                                       mss, ne):
    '''
    Generate tercile probability predictions for observed test data based on a fitted quantile regression model for a single grid point.

    Parameters:
    observed_data_train (np.ndarray): 1D array containing the observed training data for a single grid point.
    model_output_train (np.ndarray): 1D array of model output for training for a single grid point.
    observed_data_test (np.ndarray): 1D array containing the observed testing data for a single grid point.
    model_output_test (np.ndarray): 1D array of model output for testing for a single grid point.

    Returns:
    np.ndarray: A 2D array of shape (number of test data points, 3), where each element contains the probabilities of falling into the lower, middle, and upper terciles.
    '''
    
    quantiles = np.linspace(0, 100, 40)
    
    ts_grid_ij_obs_train = observed_data_train
    ts_grid_ij_model_train = model_output_train
    ts_grid_ij_obs_test = observed_data_test
    ts_grid_ij_model_test = model_output_test

    if np.all(np.isnan(ts_grid_ij_obs_train)):
        return np.zeros((ts_grid_ij_obs_test.shape[0], 3))  # Return zeros if training data is all NaNs

    X_train = ts_grid_ij_model_train.reshape(-1, 1)
    y_train = ts_grid_ij_obs_train
    X_test = ts_grid_ij_model_test.reshape(-1, 1)
    y_test = ts_grid_ij_obs_test

    # Train the model
    rfqr = RandomForestQuantileRegressor(random_state=0, min_samples_split=mss, n_estimators=ne)
    rfqr.fit(X_train, y_train)

    # Predict quantiles for test data
    pred_test_quantiles = [rfqr.predict(X_test, quantile=q).reshape(-1, 1) for q in quantiles]
    q_preds_test = np.concatenate(pred_test_quantiles, axis=1)
    
    # Construct reference CDF
    reference_cdf = np.interp(np.linspace(np.min(ts_grid_ij_obs_train), np.max(ts_grid_ij_obs_train), y_test.shape[0]), 
                              np.quantile(y_train, np.linspace(0, 100, y_train.shape[0]) / 100), 
                              np.linspace(0, 100, y_train.shape[0]) / 100, left=0, right=1)

    # Calculate individual CDFs for each test point
    individual_cdfs = np.zeros((y_test.shape[0], y_test.shape[0]))
    for k in range(y_test.shape[0]):
        xp_k = q_preds_test[k, :]
        cdf_k = np.interp(np.linspace(np.min(xp_k), np.max(xp_k), y_test.shape[0]), 
                          xp_k, np.linspace(0, 100, 40) / 100, left=0, right=1)
        individual_cdfs[k, :] = cdf_k  # CDF for each point

    p1, p2 = 0.333, 0.667

    # Calculate the absolute difference from the specific value
    absolute_difference_from_value_1 = np.abs(reference_cdf - p1)
    absolute_difference_from_value_2 = np.abs(reference_cdf - p2)

    # Find the index of the minimum difference
    index_of_value_p33_ref = np.argmin(absolute_difference_from_value_1)
    index_of_value_p67_ref = np.argmin(absolute_difference_from_value_2)

    # Obtain the tercile predictions
    triplet_array = np.zeros((y_test.shape[0], 3))
    for k in range(y_test.shape[0]):
        # Take the correct individual CDF
        p_bn = individual_cdfs[k, :][index_of_value_p33_ref]
        p_n = individual_cdfs[k, :][index_of_value_p67_ref] - p_bn
        p_an = 1 - p_n - p_bn

        triplet_array[k, 0], triplet_array[k, 1], triplet_array[k, 2] = p_bn, p_n, p_an

    return triplet_array

In [ ]:
import time

In [ ]:
mss = list(range(45, 71))
ne = list(range(25, 46))
params_d = {}

for i in range(32):
    for j in range(31):
        if np.all(np.isnan(y_train[:, i, j])):
            print(f"Skipping {i, j} as null...")
            continue
        else:
            X_train_i = X_train[:,i,j][~np.isnan(y_train[:,i,j])]
            y_train_i = y_train[:,i,j][~np.isnan(y_train[:,i,j])]
            if (np.any(np.isnan(y_val[:, i, j]))):
                X_val_i = X_val[:,i,j][~np.isnan(y_val[:,i,j])]
                y_val_i = y_val[:,i,j][~np.isnan(y_val[:,i,j])]
            else:
                X_val_i = X_val[:,i,j]
                y_val_i = y_val[:,i,j]
            print(f'Grid point: {i, j}')
            max_rpss = -np.inf
            ohe, _, _ = make_ohe(y_val_i)
            rps_c = rps_clim(ohe)

            best_params_ij = {'mss': None, 'ne': None, 'b_rpss': max_rpss}  # Initialize best_params_ij with default values
            s = time.time()

            for ms in mss:
                for n in ne:
                    tp = get_tercile_predictions_single_grid(
                        y_train_i, # y_train[:, i, j], 
                        X_train_i, # X_train[:, i, j], 
                        y_val_i, # y_val[:, i, j], 
                        X_val_i, # X_val[:, i, j], 
                        ms, n
                    )
                    rps_fc = rps_forecast(ohe, tp)
                    rpss = 1 - (rps_fc / rps_c)

                    if rpss > max_rpss:
                        max_rpss = rpss
                        best_params_ij = {'mss': ms, 'ne': n, 'b_rpss': max_rpss}
                        print(f"New best params for grid point ({i},{j}): {best_params_ij}")
                        print(f'At grid point {i},{j}, for n_estimators: {n}, min_samples_split: {ms}, the rpss is {rpss}')
                        print('-' * 115)

            if max_rpss > -np.inf:  # Ensure only grid points with valid RPSS are added
                params_d[(i, j)] = best_params_ij
                print(f"Final best params for grid point ({i},{j}): {best_params_ij}")

            e = time.time()
            print(f'Time elapsed: {e - s} seconds')
            if (i, j) in params_d:
                print(params_d.get((i, j), 0))
            print('-' * 115)

In [ ]:
import json

In [ ]:
rpss = np.full((32,31), np.nan)

In [ ]:
for i in params_d:
    rpss[i[0],i[1]] = params_d[i]['b_rpss']

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.imshow(np.flip(rpss, axis=0), vmin=-0.2, vmax=0.2, cmap='bwr')

In [ ]:
params_d1 = {str(k):v for k,v in params_d.items()}

In [ ]:
with open('best_params_2_randomized.json', 'w') as json_file:
    json.dump(params_d1, json_file)

In [ ]:
import json
import ast

# Read dictionary from a JSON file
with open('best_params_2_randomized.json', 'r') as json_file:
    params_str_keys = json.load(json_file)

# Convert string keys back to tuples
best_params_prop = {ast.literal_eval(k): v for k, v in params_str_keys.items()}

In [ ]:
best_params_prop

In [ ]:
y_test.shape

In [ ]:
y_val.shape

In [ ]:
y_train.shape

In [ ]:
from tqdm import tqdm

In [ ]:
new_rpss_mat = np.full((32,31), np.nan)
terciles = np.full((32,31,65,3), np.nan)
ohe_mat = np.zeros((32,31,65,3))
rps_fc_mat = np.full((32,31),np.nan)
for i in tqdm(best_params_prop):

    try:
        if np.all(np.isnan(y_train[:, i[0], i[1]])):
            print(f"Skipping {i[0], i[1]} as null...")
            continue
        ohe, _, _ = make_ohe(y_test[:, i[0], i[1]])
        ohe_mat[i[0],i[1],:,:] = ohe
        rps_c = rps_clim(ohe)
        ms, n = best_params_prop[(i[0],i[1])]['mss'], best_params_prop[(i[0],i[1])]['ne']
    #     tp = get_tercile_predictions_single_grid(
    #         train_obs_values[:, i[0], i[1]], 
    #         train_model_values[:, i[0], i[1]], 
    #         test_obs_values[:, i[0], i[1]], 
    #         test_model_values[:, i[0], i[1]], ms, n
    #     )
        tp = get_tercile_predictions_single_grid(
                            y_train[:,i[0],i[1]], # y_train[:, i, j], 
                            X_train[:,i[0],i[1]], # X_train[:, i, j], 
                            y_test[:,i[0],i[1]], # y_val[:, i, j], 
                            X_test[:,i[0],i[1]], # X_val[:, i, j], 
                            ms, n)
        terciles[i[0],i[1],:,:] = tp
        rps_fc = rps_forecast(ohe, tp)
        rps_fc_mat[i[0],i[1]] = rps_fc
        new_rpss_mat[i[0], i[1]] = 1 - (rps_fc / rps_c)
    except:
        print(f"Skipping {i[0], i[1]} as it contains nan values...")
        continue

In [ ]:
plt.imshow(np.flip(new_rpss_mat, axis=0), cmap='bwr', vmin=-0.2, vmax=0.2)
plt.colorbar()

In [ ]:
len(best_params_prop)

In [ ]:
non_null_points = len(best_params_prop)-22

In [ ]:
non_null_points

In [ ]:
np.save('rpss_2_randomized_all_samples.npy', new_rpss_mat)